In [1]:
import sys
import os
import time
import uuid
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Setup context
from dotenv import load_dotenv
load_dotenv()
from Tool.builtin.memorytool import register_memory_tools
from core.llm import EasyLLM
from agent.BasicAgent import BasicAgent
from memory.V2.MemoryManage import MemoryManage
from memory.V2.BaseMemory import MemoryConfig
from memory.V2.WorkingMemory import WorkingMemory
from memory.V2.EpisodicMemory import EpisodicMemory
from memory.V2.PerceptualMemory import PerceptualMemory
from memory.V2.SemanticMemory import SemanticMemory
from memory.V2.Store.Neo4jGraphStore import Neo4jGraphStore
from memory.V2.Extractor.Extractor import Extractor

from Tool.ToolRegistry import ToolRegistry
from memory.V2.Store.SQLiteDocumentStore import SQLiteDocumentStore
from memory.V2.Store.QdrantVectorStore import QdrantVectorStore
from memory.V2.Embedding.HuggingfaceEmbeddingModel import HuggingfaceEmbeddingModel



/home/wxd/miniconda3/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
class TestAgentMemoryIntegration:
    def __init__(self):
        print("========== 初始化测试环境 ==========")
        self.config = MemoryConfig(max_capacity=20)
        
        # 使用基于内存的 SQLite 和 Qdrant 以防无相关环境依赖
        self.embedding_model = HuggingfaceEmbeddingModel(model_name="sentence-transformers/all-MiniLM-L6-v2")
        self.working_memory = WorkingMemory(self.config, self.embedding_model)
        
        self.episodic_doc_store = SQLiteDocumentStore(db_path=":memory:")
        self.episodic_vector_store = QdrantVectorStore(way="memory", collection_name="test_ep_integration")
        self.episodic_memory = EpisodicMemory(
            config=self.config,
            document_store=self.episodic_doc_store,
            vector_store=self.episodic_vector_store,
            embedding_model=self.embedding_model
        )
        # 简单实例化 perceptual 依赖
        self.perceptual_doc_store = SQLiteDocumentStore(db_path="/tmp/test_tool_perceptual.db")
        self.perceptual_vector_store = QdrantVectorStore(way="memory", collection_name="test_tool_pe", vector_size=384)
        self.perceptual_image_store = QdrantVectorStore(way="memory", collection_name="test_tool_pe_image", vector_size=512)
        self.perceptual_memory = PerceptualMemory(
            memory_config=self.config,
            document_store=self.perceptual_doc_store,
            vector_stores={"text": self.perceptual_vector_store, "image": self.perceptual_image_store},
            embedding_model=self.embedding_model,
            supported_modalities=["text", "image"]
        )

        # 简单实例化 semantic 依赖
        self.graph_store = Neo4jGraphStore(uri="bolt://localhost:17687", username="neo4j", password="password")
        self.vector_store = QdrantVectorStore(way="memory", collection_name="semantic_memory")
        self.llm = EasyLLM() # Assume valid config internally
        self.extractor = Extractor(self.llm, False)
        self.semantic_memory = SemanticMemory(
            memory_config=MemoryConfig(),
            vector_store=self.vector_store,
            graph_store=self.graph_store,
            extractor=self.extractor,
            embedding_model=self.embedding_model
        )
        # 初始化 MemoryManage，为了简单只开启 working 和 episodic
        self.mm = MemoryManage(
            config=self.config,
            user_id="test_integration_user",
            enable_working=True,
            working_memory=self.working_memory,
            enable_episodic=True,
            episodic_memory=self.episodic_memory,
            enable_semantic=True,
            semantic_memory=self.semantic_memory,
            enable_perceptual=True,
            perceptual_memory=self.perceptual_memory
        ) 
        
        self.tool_registry = ToolRegistry()
        
        # 这里验证 Agent 是否能和 MemoryManage 自动结合（.with_memory或初始化传入）
        self.agent = BasicAgent(
            name="IntegrationTestAgent",
            llm=self.llm,
            system_prompt="你是一个具有记忆系统的测试AI。请按照用户的指令进行操作。",
            enable_tool=True,
            tool_registry=self.tool_registry,
            verbose_thinking=False
        ).with_memory(self.mm)
        
        # .with_memory 应该自动注册了工具
        tools = self.tool_registry.get_openai_tools()
        assert any(t["function"]["name"] == "add_memory_tool" for t in tools), "Tool registry failed to auto-register memory tools!"
        
        print("测试环境初始化完成！")

    def test_implicit_prompt_injection(self):
        print("\n========== 测试1: 潜意识记忆注入 ==========")
        self.agent.clear_history()
        self.mm.clear_memories()
        
        # 我们手动向 Working Memory 添加一条便签
        self.mm.add_memory(content="系统的核心密码是：42", memory_type="working", importance=1.0)
        
        # 问 Agent 这个信息，Agent 虽然没有历史，但是 system prompt 里有它的 inject 
        prompt = "系统的核心密码是多少？不要调用任何工具，只能依靠你的已有记忆。"
        print(f"User: {prompt}")
        
        response = self.agent.invoke(prompt)
        print(f"Agent: {response}")
        
        if "42" in response:
            print("✅ 潜意识记忆注入测试通过！")
        else:
            print("❌ 潜意识记忆注入测试失败！")

    def test_agent_tool_usage_for_memory(self):
        print("\n========== 测试2: 智能调用工具读写记忆 ==========")
        self.agent.clear_history()
        self.mm.clear_memories()
        
        # 不再明确指示要求使用工具和指定记忆类型，期待 Agent 自动判断
        prompt1 = "我告诉你一个重要的秘密：昨天小明给了我一个苹果。"
        print(f"User: {prompt1}")
        res1 = self.agent.invoke(prompt1)
        print(f"Agent: {res1}")
        
        # 检查是否确实存入了 episodic 或 working
        ep_memories = self.mm.memory_types["episodic"].get_all_memories()
        working_memories = self.mm.memory_types["working"].get_all_memories()
        assert len(ep_memories) > 0 or len(working_memories) > 0, "Agent未能成功调用工具存入记忆"
        print("✅ 成功发现智能存入的新记忆")
        
        # 触发搜索
        self.agent.clear_history()
        prompt2 = "昨天小明给了我什么东西？"
        print(f"User: {prompt2}")
        res2 = self.agent.invoke(prompt2)
        print(f"Agent: {res2}")
        
        if "苹果" in res2:
            print("✅ 智能检索记忆测试通过！")
        else:
            print("❌ 智能检索记忆测试失败！")
            
    def test_memory_update_and_remove(self):
        print("\n========== 测试4: 智能调用工具更新和删除记忆 ==========")
        self.agent.clear_history()
        self.mm.clear_memories()
        
        prompt1 = "我最喜欢的水果是香蕉，请一定要记住。"
        print(f"User: {prompt1}")
        self.agent.invoke(prompt1)
        
        prompt2 = "我口味变了，我现在最喜欢的水果是苹果，不再是香蕉了"
        print(f"User: {prompt2}")
        res2 = self.agent.invoke(prompt2)
        print(f"Agent: {res2}")
        
        self.agent.clear_history()
        prompt3 = "我现在最喜欢的水果是什么？"
        res3 = self.agent.invoke(prompt3)
        print(f"Agent: {res3}")
        
        if "苹果" in res3 and "香蕉" not in res3:
            print("✅ 记忆更新测试通过！")
        else:
            print("❌ 记忆更新测试不完全通过或失败！答案中包含: " + res3)
            
        prompt4 = "刚才说的关于我最喜欢吃什么水果的事情，全部忘掉吧"
        print(f"User: {prompt4}")
        self.agent.invoke(prompt4)
        
        self.agent.clear_history()
        prompt5 = "我最喜欢吃什么水果？"
        res5 = self.agent.invoke(prompt5)
        print(f"Agent: {res5}")
        
        # Agent 应该回答不知道或没有相关记录
        if "苹果" not in res5 and "香蕉" not in res5:
            print("✅ 记忆删除测试通过！")
        else:
            print("❌ 记忆删除测试失败！答案中仍有相关信息: " + res5)
            
    def test_background_auto_extraction(self):
        print("\n========== 测试3: 后台自动记忆提炼机制 ==========")
        self.agent.clear_history()
        self.mm.clear_memories()
        
        current_episodic_count = len(self.mm.memory_types["episodic"].get_all_memories())
        current_working_count = len(self.mm.memory_types["working"].get_all_memories())
        
        # 模拟频繁对话，达到 _unextracted_msg_count >= 5 阈值
        print("开始输入 5 轮闲聊对话...")
        for i in range(10):
            msg = f"这是第 {i+1} 轮闲聊内容，我很喜欢吃水果，特别是香蕉"
            print(f"User: {msg}")
            # 注意: add_message 或者 add_user_message 都会触发检查
            self.agent.add_user_message(msg)
            self.agent.add_assistant_message(f"好的，我收到了你的第 {i+1} 轮消息。")
            
        print("等待后台提炼线程执行 (约7秒)...")
        time.sleep(20) # 预留时间给守护线程执行 LLM
        
        # 验证是否有新的记忆被自动注入到系统中
        new_episodic_count = len(self.mm.memory_types["episodic"].get_all_memories())
        new_working_count = len(self.mm.memory_types["working"].get_all_memories())
        
        total_new_memories = (new_episodic_count - current_episodic_count) + (new_working_count - current_working_count)
        
        if total_new_memories > 0:
            print(f"✅ 后台自动提炼机制测试通过！新增了 {total_new_memories} 条记忆。")
            # 打印这些新记忆
            print(self.mm.get_all_memories())
        else:
            print("❌ 未观察到自动提炼产生新的记忆，可能是LLM抽取失败或线程未执行。")


    def test_working_memory_lifecycle(self):
        print("========== 测试5: Working Memory (便签本) 自动化管理 ==========")
        self.agent.clear_history()
        self.mm.clear_memories()
        
        # 1. 测试主动添加：给一个包含多个约束的任务，看 Agent 是否记录到 working memory
        prompt1 = "我们要策划一个50人的聚餐，预算是5000元，地点定在海鲜饭店。"
        print(f"User: {prompt1}")
        self.agent.invoke(prompt1)
        
        wm_memories = self.mm.memory_types["working"].get_all_memories()
        has_constraints = any("预算" in m.content or "5000" in m.content for m in wm_memories)
        if has_constraints:
            print(f"✅ 成功自动记录约束到 Working Memory: {[m.content for m in wm_memories]}")
        else:
            print("❌ 未能在 Working Memory 中发现约束记录")
            
        # 2. 测试更新：修改其中一个约束
        prompt2 = "计划有变，人数增加到80人，预算提高到8000元。"
        print(f"User: {prompt2}")
        self.agent.invoke(prompt2)
        
        wm_memories = self.mm.memory_types["working"].get_all_memories()
        is_updated = any("8000" in m.content or "80" in m.content for m in wm_memories)
        if is_updated:
            print(f"✅ Working Memory 已成功更新: {[m.content for m in wm_memories]}")
        else:
            print("❌ Working Memory 更新失败")
            
        # 3. 测试删除：完成任务后清理
        prompt3 = "聚餐策划取消了"
        print(f"User: {prompt3}")
        self.agent.invoke(prompt3)
        
        wm_memories = self.mm.memory_types["working"].get_all_memories()
        if len(wm_memories) == 0:
            print("✅ Working Memory 已成功清空记录")
        else:
            print(f"⚠️ Working Memory 仍有残留: {[m.content for m in wm_memories]}")
    
    def test_no_use_memory(self):
        print("========== 测试6: 不使用记忆 ==========")
        self.agent.clear_history()
        self.mm.clear_memories()
        
        prompt1 = "解释一下什么是RAG"
        print(f"User: {prompt1}")
        self.agent.invoke(prompt1)
        
        wm_memories = self.mm.get_all_memories()
        if len(wm_memories) == 0:
            print("✅ Working Memory 已成功清空记录")
        else:
            print(f"⚠️ Working Memory 仍有残留: {[m.content for m in wm_memories]}")
    def run_all(self):
        try:
            # self.test_implicit_prompt_injection()
            # self.test_agent_tool_usage_for_memory()
            # self.test_background_auto_extraction()
            self.test_memory_update_and_remove()
            self.test_working_memory_lifecycle()
            print("\n🏁 所有Agent-Memory集成测试结束！")
        except Exception as e:
            print(f"\n❌ 测试过程中发生异常: {e}")

In [ ]:
test_agent_memory=TestAgentMemoryIntegration()


INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


========== 初始化测试环境 ==========


INFO:memory.V2.Store.QdrantVectorStore:✅成功连接到Qdrant内存
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功创建集合: test_ep_integration
INFO:memory.V2.Store.QdrantVectorStore:✅成功连接到Qdrant内存
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功创建集合: test_tool_pe
INFO:memory.V2.Store.QdrantVectorStore:✅成功连接到Qdrant内存
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功创建集合: test_tool_pe_image


In [ ]:
test_agent_memory.test_no_use_memory()

In [10]:
test_agent_memory.test_working_memory_lifecycle()

INFO:core.agent:对话历史已清空
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功删除episodic类型的向量
INFO:memory.V2.SemanticMemory:✅ 向量数据库已清空
INFO:memory.V2.SemanticMemory:✅ 图数据库已清空
INFO:memory.V2.SemanticMemory:✅ 所有语义记忆已清空
INFO:memory.V2.PerceptualMemory:✅ 成功清空文档存储中的感知记忆
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功删除perceptual类型的向量
INFO:memory.V2.PerceptualMemory:✅ 成功清空text向量存储中的感知记忆
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功删除perceptual类型的向量
INFO:memory.V2.PerceptualMemory:✅ 成功清空image向量存储中的感知记忆
INFO:agent.BasicAgent:使用工具模式调用智能体


========== 测试5: Working Memory (便签本) 自动化管理 ==========
User: 我们要策划一个50人的聚餐，预算是5000元，地点定在海鲜饭店。


INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
INFO:core.providers.google_provider:✅ Google Provider 工具调用响应成功
INFO:agent.BasicAgent:执行工具: add_memory_tool，参数: {'content': '50人聚餐策划案：\n- 参与人数：50人\n- 总预算：5000元（人均100元）\n- 餐厅类型：海鲜饭店\n- 任务：策划聚餐细节（菜单、座位、流程等）', 'importance': 0.8, 'memory_type': 'working'}
INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
INFO:core.providers.google_provider:✅ Google Provider 工具调用响应成功
INFO:agent.BasicAgent:使用工具模式调用智能体


✅ 成功自动记录约束到 Working Memory: ['50人聚餐策划案：\n- 参与人数：50人\n- 总预算：5000元（人均100元）\n- 餐厅类型：海鲜饭店\n- 任务：策划聚餐细节（菜单、座位、流程等）']
User: 计划有变，人数增加到80人，预算提高到8000元。


INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
INFO:core.providers.google_provider:✅ Google Provider 工具调用响应成功
INFO:agent.BasicAgent:执行工具: update_memory_tool，参数: {'content': '80人聚餐策划案：\n- 参与人数：80人\n- 总预算：8000元（人均100元）\n- 餐厅类型：海鲜饭店\n- 任务：策划聚餐细节（菜单、座位、流程等）', 'memory_id': '7eb97243-0b08-48d5-bdcb-a82e52876aae'}
INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
INFO:core.providers.google_provider:✅ Google Provider 工具调用响应成功
INFO:agent.BasicAgent:使用工具模式调用智能体


✅ Working Memory 已成功更新: ['80人聚餐策划案：\n- 参与人数：80人\n- 总预算：8000元（人均100元）\n- 餐厅类型：海鲜饭店\n- 任务：策划聚餐细节（菜单、座位、流程等）']
User: 聚餐策划取消了


INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
INFO:core.providers.google_provider:✅ Google Provider 工具调用响应成功
INFO:agent.BasicAgent:执行工具: remove_memory_tool，参数: {'memory_id': '7eb97243-0b08-48d5-bdcb-a82e52876aae'}
INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
INFO:core.providers.google_provider:✅ Google Provider 工具调用响应成功


✅ Working Memory 已成功清空记录


In [ ]:
test_agent_memory.test_implicit_prompt_injection()
print(test_agent_memory.agent.invoke("系统的核心密码是多少？"))

In [5]:
test_agent_memory.test_agent_tool_usage_for_memory()

INFO:core.agent:对话历史已清空
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功删除episodic类型的向量
INFO:memory.V2.SemanticMemory:✅ 向量数据库已清空
INFO:memory.V2.SemanticMemory:✅ 图数据库已清空
INFO:memory.V2.SemanticMemory:✅ 所有语义记忆已清空
INFO:memory.V2.PerceptualMemory:✅ 成功清空文档存储中的感知记忆
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功删除perceptual类型的向量
INFO:memory.V2.PerceptualMemory:✅ 成功清空text向量存储中的感知记忆
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功删除perceptual类型的向量
INFO:memory.V2.PerceptualMemory:✅ 成功清空image向量存储中的感知记忆
INFO:agent.BasicAgent:使用工具模式调用智能体



========== 测试2: 智能调用工具读写记忆 ==========
User: 我告诉你一个重要的秘密：昨天小明给了我一个苹果。


INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
INFO:core.providers.google_provider:✅ Google Provider 工具调用响应成功
INFO:agent.BasicAgent:执行工具: add_memory_tool，参数: {'content': '昨天小明给了用户一个苹果。', 'importance': 0.8, 'memory_type': 'episodic', 'metadata': {'source': 'user_conversation'}}
Batches: 100%|██████████| 1/1 [00:00<00:00, 25.15it/s]
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功添加1个向量
INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
INFO:core.providers.google_provider:✅ Google Provider 工具调用响应成功
INFO:core.agent:对话历史已清空
INFO:agent.BasicAgent:使用工具模式调用智能体


Agent: OK，我已经把这个“重要的秘密”记在心里了（已存入情景记忆）。我会记得昨天小明给了你一个苹果。
✅ 成功发现智能存入的新记忆
User: 昨天小明给了我什么东西？


INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
INFO:core.providers.google_provider:✅ Google Provider 工具调用响应成功
INFO:agent.BasicAgent:执行工具: search_memory_tool，参数: {'memory_types': ['episodic'], 'query': '小明给了我什么'}
Batches: 100%|██████████| 1/1 [00:00<00:00, 29.15it/s]
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功搜索到1个向量
INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
INFO:core.providers.google_provider:✅ Google Provider 工具调用响应成功


Agent: 昨天小明给了你一个苹果。
✅ 智能检索记忆测试通过！


In [6]:
test_agent_memory.test_background_auto_extraction()

INFO:core.agent:对话历史已清空
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功删除episodic类型的向量
INFO:memory.V2.SemanticMemory:✅ 向量数据库已清空
INFO:memory.V2.SemanticMemory:✅ 图数据库已清空
INFO:memory.V2.SemanticMemory:✅ 所有语义记忆已清空
INFO:memory.V2.PerceptualMemory:✅ 成功清空文档存储中的感知记忆
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功删除perceptual类型的向量
INFO:memory.V2.PerceptualMemory:✅ 成功清空text向量存储中的感知记忆
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功删除perceptual类型的向量
INFO:memory.V2.PerceptualMemory:✅ 成功清空image向量存储中的感知记忆
INFO:core.agent:启动后台记忆提炼 (Background Memory Extraction)...
INFO:agent.BasicAgent:BasicAgent 'MemoryExtractor' 初始化完成，工具调用: 启用，异步执行: 禁用，provider: google
INFO:agent.BasicAgent:使用工具模式调用智能体



========== 测试3: 后台自动记忆提炼机制 ==========
开始输入 5 轮闲聊对话...
User: 这是第 1 轮闲聊内容，我很喜欢吃水果，特别是香蕉
User: 这是第 2 轮闲聊内容，我很喜欢吃水果，特别是香蕉
User: 这是第 3 轮闲聊内容，我很喜欢吃水果，特别是香蕉
User: 这是第 4 轮闲聊内容，我很喜欢吃水果，特别是香蕉
User: 这是第 5 轮闲聊内容，我很喜欢吃水果，特别是香蕉
等待后台提炼线程执行 (约7秒)...


INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
INFO:core.providers.google_provider:✅ Google Provider 工具调用响应成功
INFO:agent.BasicAgent:执行工具: add_memory_tool，参数: {'content': '用户喜欢吃水果，特别是香蕉。', 'importance': 0.8, 'memory_type': 'semantic'}
Batches: 100%|██████████| 1/1 [00:00<00:00, 18.20it/s]
INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
INFO:core.providers.google_provider:✅ Google Provider 响应成功
INFO:agent.StructuredOutputAgent:结构化输出解析成功 (尝试 1/3)
INFO:memory.V2.Extractor.Extractor:提取完成: 3 个实体, 3 个关系
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功添加1个向量
INFO:memory.V2.Store.Neo4jGraphStore:已添加实体: 用户 (人物)
INFO:memory.V2.SemanticMemory:添加新实体: 93b97cd2-66f9-4d1d-a73e-edb2503b369d
INFO:memory.V2.Store.Neo4jGraphStore:已添加实体: 水果 (概念)
INFO:memory.V2.SemanticMemory:添加新实体: 1e03edfc-9beb-429e-8637-fb4572746814
INFO:memory.V2.Store.Neo4jGraphStore:已添加实体: 香蕉 (概念)
INFO:memory.V2.SemanticMemory:添加新实体: 6f4e8e50-71b4-

Failed to add memory: SQLite objects created in a thread can only be used in that same thread. The object was created in thread id 140389347738688 and this is thread id 140357243237952.


Batches: 100%|██████████| 1/1 [00:00<00:00, 24.86it/s]
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功添加1个向量
INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
INFO:core.providers.google_provider:✅ Google Provider 工具调用响应成功
INFO:core.agent:后台记忆提炼完成，对话已被 LLM 自主归档。


✅ 后台自动提炼机制测试通过！新增了 1 条记忆。
[MemoryItem(id='3c9b2bf4-c2a1-4ff6-af21-660b014fb7d8', content='用户在连续5轮对话中反复强调了自己喜欢吃香蕉。', type='episodic', user_id='test_integration_user', timestamp=datetime.datetime(2026, 3, 12, 15, 44, 40, 830925), importance=0.5, metadata={'session_id': 'session_20260312_153946', 'outcome': None, 'context': {}}), MemoryItem(id='76b6b70f-d88a-47d0-b628-4d80fbced386', content='用户喜欢吃水果，特别是香蕉。', type='semantic', user_id='test_integration_user', timestamp=datetime.datetime(2026, 3, 12, 15, 44, 30, 412934), importance=0.8, metadata={'session_id': 'session_20260312_153946', 'conversation_count': 2, 'timestamp': '2026-03-12T15:44:30.412443', 'entities': ['93b97cd2-66f9-4d1d-a73e-edb2503b369d', '1e03edfc-9beb-429e-8637-fb4572746814', '6f4e8e50-71b4-4cd0-82b0-5b05b53fc968'], 'relations': ['93b97cd2-66f9-4d1d-a73e-edb2503b369d|||喜欢吃|||1e03edfc-9beb-429e-8637-fb4572746814', '93b97cd2-66f9-4d1d-a73e-edb2503b369d|||特别喜欢吃|||6f4e8e50-71b4-4cd0-82b0-5b05b53fc968', '6f4e8e50-71b4-4cd0-82b0

In [7]:
test_agent_memory.test_memory_update_and_remove()

INFO:core.agent:对话历史已清空
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功删除episodic类型的向量
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功删除1个向量
INFO:memory.V2.SemanticMemory:✅ 向量数据库已清空
INFO:memory.V2.SemanticMemory:✅ 图数据库已清空
INFO:memory.V2.SemanticMemory:✅ 所有语义记忆已清空
INFO:memory.V2.PerceptualMemory:✅ 成功清空文档存储中的感知记忆
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功删除perceptual类型的向量
INFO:memory.V2.PerceptualMemory:✅ 成功清空text向量存储中的感知记忆
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功删除perceptual类型的向量
INFO:memory.V2.PerceptualMemory:✅ 成功清空image向量存储中的感知记忆
INFO:agent.BasicAgent:使用工具模式调用智能体



========== 测试4: 智能调用工具更新和删除记忆 ==========
User: 我最喜欢的水果是香蕉，请一定要记住。


INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
INFO:core.providers.google_provider:✅ Google Provider 工具调用响应成功
INFO:agent.BasicAgent:执行工具: add_memory_tool，参数: {'content': '用户最喜欢的水果是香蕉。', 'importance': 0.8, 'memory_type': 'episodic'}
Batches: 100%|██████████| 1/1 [00:00<00:00, 19.14it/s]
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功添加1个向量
INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
INFO:core.providers.google_provider:✅ Google Provider 工具调用响应成功
INFO:agent.BasicAgent:使用工具模式调用智能体


User: 我口味变了，我现在最喜欢的水果是苹果，不再是香蕉了，请更新你的记忆。


INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
INFO:core.providers.google_provider:✅ Google Provider 工具调用响应成功
INFO:agent.BasicAgent:执行工具: search_memory_tool，参数: {'query': '最喜欢的水果'}
Batches: 100%|██████████| 1/1 [00:00<00:00, 28.51it/s]
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功搜索到1个向量
Batches: 100%|██████████| 1/1 [00:00<00:00, 28.40it/s]
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功搜索到0个向量
INFO:memory.V2.SemanticMemory:🔍 [DEBUG] 向量搜索返回 0 条结果
INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
INFO:core.providers.google_provider:✅ Google Provider 响应成功
INFO:agent.StructuredOutputAgent:结构化输出解析成功 (尝试 1/3)
INFO:memory.V2.Extractor.Extractor:提取完成: 2 个实体, 1 个关系
INFO:memory.V2.SemanticMemory:🔍 [DEBUG] 提取到 2 个查询实体: ['水果', '最喜欢']
INFO:memory.V2.SemanticMemory:🔍 [DEBUG] entities_name_to_id 缓存: []
INFO:memory.V2.SemanticMemory:🔍 [DEBUG] 实体 '水果' get_related_entities 返回 0 个
INFO:memory.V2.SemanticMemory:🔍 [DEBUG

Agent: OK，我已经更新了记忆：你现在最喜欢的水果是苹果，不再是香蕉了。🍎


INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
INFO:core.providers.google_provider:✅ Google Provider 工具调用响应成功
INFO:agent.BasicAgent:执行工具: search_memory_tool，参数: {'query': '最喜欢的水果'}
Batches: 100%|██████████| 1/1 [00:00<00:00, 26.50it/s]
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功搜索到1个向量
Batches: 100%|██████████| 1/1 [00:00<00:00, 30.03it/s]
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功搜索到0个向量
INFO:memory.V2.SemanticMemory:🔍 [DEBUG] 向量搜索返回 0 条结果
INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
INFO:core.providers.google_provider:✅ Google Provider 响应成功
INFO:agent.StructuredOutputAgent:结构化输出解析成功 (尝试 1/3)
INFO:memory.V2.Extractor.Extractor:提取完成: 2 个实体, 1 个关系
INFO:memory.V2.SemanticMemory:🔍 [DEBUG] 提取到 2 个查询实体: ['水果', '最喜欢']
INFO:memory.V2.SemanticMemory:🔍 [DEBUG] entities_name_to_id 缓存: []
INFO:memory.V2.SemanticMemory:🔍 [DEBUG] 实体 '水果' get_related_entities 返回 0 个
INFO:memory.V2.SemanticMemory:🔍 [DEBUG

Agent: 你现在最喜欢的水果是**苹果**。

根据我的记录，你之前可能喜欢香蕉，但现在已经改为最喜欢苹果了。
❌ 记忆更新测试不完全通过或失败！答案中包含: 你现在最喜欢的水果是**苹果**。

根据我的记录，你之前可能喜欢香蕉，但现在已经改为最喜欢苹果了。
User: 刚才说的关于我最喜欢吃什么水果的事情，全部忘掉吧，把那条记忆删了。


INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
INFO:core.providers.google_provider:✅ Google Provider 工具调用响应成功
INFO:agent.BasicAgent:执行工具: search_memory_tool，参数: {'query': '最喜欢的水果'}
Batches: 100%|██████████| 1/1 [00:00<00:00, 29.51it/s]
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功搜索到1个向量
Batches: 100%|██████████| 1/1 [00:00<00:00, 32.21it/s]
INFO:memory.V2.Store.QdrantVectorStore:✅ 成功搜索到0个向量
INFO:memory.V2.SemanticMemory:🔍 [DEBUG] 向量搜索返回 0 条结果
INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
INFO:core.providers.google_provider:✅ Google Provider 响应成功
INFO:agent.StructuredOutputAgent:结构化输出解析成功 (尝试 1/3)
INFO:memory.V2.Extractor.Extractor:提取完成: 2 个实体, 1 个关系
INFO:memory.V2.SemanticMemory:🔍 [DEBUG] 提取到 2 个查询实体: ['水果', '最喜欢']
INFO:memory.V2.SemanticMemory:🔍 [DEBUG] entities_name_to_id 缓存: []
INFO:memory.V2.SemanticMemory:🔍 [DEBUG] 实体 '水果' get_related_entities 返回 0 个
INFO:memory.V2.SemanticMemory:🔍 [DEBUG

Agent: 你最喜欢吃的水果我目前还不清楚。由于我的记忆库中没有找到相关的记录，你可以告诉我你最喜欢吃的水果，这样我就能记住它，以后你再问我时我就能回答你了。
✅ 记忆删除测试通过！
